# Коллаборативные модели: от лога до честного бейзлайна

ML Mentor · starter notebook · решения в репозитории отсутствуют.


## Что предстоит сделать

За 90 минут вы пройдёте путь от лога лайков до решения, которое можно защищать перед командой: построите популярность, item-to-item и implicit ALS, сравните их на одном временном разбиении и отдельно проверите сложные срезы. Обязательный путь рассчитан на CPU и должен выполняться не более 15 минут.

**Итог:** одна таблица с Recall@20, NDCG@20, покрытием каталога, временем обучения и применения; отдельные метрики для пользователей с короткой историей и редких объектов; решение, какой бейзлайн выпускать первым.

Случайное разбиение запрещено. Уже просмотренные объекты нужно удалить из рекомендаций. Фраза «ALS лучше» без разбора срезов и стоимости не считается инженерным выводом.

### На чём основана работа

Последовательность экспериментов опирается на практику [недели 1](https://github.com/yandexdataschool/recsys_course/blob/2026_spring/week01_intro/practice.ipynb) и [недели 3](https://github.com/yandexdataschool/recsys_course/blob/2026_spring/week03_ranking/practice/practice.ipynb) курса ШАД RecSys 2026. Текст, код и проверки написаны заново для ML Mentor. Данные — детерминированная выборка Yambda-50M, Apache 2.0; ревизия, преобразование и контрольные суммы описаны в `data/datasets.json`.

In [ ]:
import hashlib
import subprocess
import sys
import time
import urllib.request
from importlib.metadata import PackageNotFoundError, version as installed_version
from pathlib import Path
from urllib.parse import urlparse


REQUIRED_PACKAGES = {
    "pyarrow": "22.0.0",
    "implicit": "0.7.3",
    "catboost": "1.2.10",
}


def install_missing_packages() -> None:
    missing = []
    for package, version in REQUIRED_PACKAGES.items():
        try:
            actual = installed_version(package)
        except PackageNotFoundError:
            missing.append(f"{package}=={version}")
            continue
        if actual != version:
            missing.append(f"{package}=={version}")
    if missing:
        subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", *missing], check=True)


install_missing_packages()

import numpy as np
import pandas as pd
import pyarrow
import implicit
import catboost
from scipy import sparse


SEED = 42
FIXTURE_URL = "https://raw.githubusercontent.com/Vosatorp/ml-mentor-labs/a8542e1f42cecc21a1a7278955b6fd323ced8c9b/data/yambda/yambda-50m-likes-compact.parquet"
FIXTURE_SHA256 = "ca4dbdffea7114131ad9da411419838ed9c0a830613de935deaeecaad14d4805"
FIXTURE_BYTES = 2031420
EXPECTED_COLUMNS = ["uid", "item_id", "timestamp", "is_organic"]


def sha256(payload: bytes) -> str:
    return hashlib.sha256(payload).hexdigest()


def require_sha256(payload: bytes, expected: str, label: str) -> None:
    actual = sha256(payload)
    if actual != expected:
        raise RuntimeError(f"{label}: checksum changed: {actual}")


def require_final_hostname(response, expected: str) -> None:
    actual = urlparse(response.geturl()).hostname
    if actual != expected:
        raise RuntimeError(f"unexpected redirect host: {actual}")


def resolve_fixture_path() -> Path:
    for root in [Path.cwd(), *Path.cwd().parents]:
        candidate = root / "data" / "yambda" / "yambda-50m-likes-compact.parquet"
        if candidate.is_file():
            payload = candidate.read_bytes()
            if len(payload) != FIXTURE_BYTES:
                raise RuntimeError(f"fixture size changed: {len(payload)}")
            require_sha256(payload, FIXTURE_SHA256, "local Yambda fixture")
            return candidate

    cache_path = Path("/tmp/ml-mentor-labs-datasets-v1/yambda/yambda-50m-likes-compact.parquet")
    if cache_path.is_file():
        payload = cache_path.read_bytes()
        if len(payload) != FIXTURE_BYTES:
            raise RuntimeError(f"cached fixture size changed: {len(payload)}")
        require_sha256(payload, FIXTURE_SHA256, "cached Yambda fixture")
        return cache_path

    request = urllib.request.Request(FIXTURE_URL, headers={"User-Agent": "ML-Mentor-Labs/1.0"})
    with urllib.request.urlopen(request, timeout=120) as response:
        require_final_hostname(response, "raw.githubusercontent.com")
        payload = response.read()
    if len(payload) != FIXTURE_BYTES:
        raise RuntimeError(f"downloaded fixture size changed: {len(payload)}")
    require_sha256(payload, FIXTURE_SHA256, "downloaded Yambda fixture")
    cache_path.parent.mkdir(parents=True, exist_ok=True)
    cache_path.write_bytes(payload)
    return cache_path


FIXTURE_PATH = resolve_fixture_path()
events = pd.read_parquet(FIXTURE_PATH, columns=EXPECTED_COLUMNS)
events = events.sort_values(["timestamp", "uid", "item_id"], kind="mergesort").reset_index(drop=True)
if list(events.columns) != EXPECTED_COLUMNS:
    raise RuntimeError(f"unexpected columns: {list(events.columns)}")
if len(events) != 268_631 or events.uid.nunique() != 1_999 or events.item_id.nunique() != 42_965:
    raise RuntimeError("unexpected fixture statistics")
if not events.timestamp.is_monotonic_increasing:
    raise RuntimeError("events must be sorted by time")

TRAIN_BOUNDARY = int(events.timestamp.quantile(0.70, interpolation="lower"))
VALIDATION_BOUNDARY = int(events.timestamp.quantile(0.85, interpolation="lower"))
train_events = events[events.timestamp <= TRAIN_BOUNDARY].copy()
validation_events = events[(events.timestamp > TRAIN_BOUNDARY) & (events.timestamp <= VALIDATION_BOUNDARY)].copy()
test_events = events[events.timestamp > VALIDATION_BOUNDARY].copy()

USER_IDS = np.sort(train_events.uid.unique())
ITEM_IDS = np.sort(train_events.item_id.unique())
USER_TO_INDEX = {int(value): index for index, value in enumerate(USER_IDS)}
ITEM_TO_INDEX = {int(value): index for index, value in enumerate(ITEM_IDS)}
INDEX_TO_ITEM = np.asarray(ITEM_IDS)


def to_user_item_matrix(frame: pd.DataFrame) -> sparse.csr_matrix:
    known = frame[frame.uid.isin(USER_TO_INDEX) & frame.item_id.isin(ITEM_TO_INDEX)]
    rows = known.uid.map(USER_TO_INDEX).to_numpy()
    columns = known.item_id.map(ITEM_TO_INDEX).to_numpy()
    values = np.ones(len(known), dtype=np.float32)
    matrix = sparse.csr_matrix((values, (rows, columns)), shape=(len(USER_IDS), len(ITEM_IDS)))
    matrix.data[:] = 1.0
    matrix.eliminate_zeros()
    return matrix


train_matrix = to_user_item_matrix(train_events)
print({
    "versions": {"pyarrow": pyarrow.__version__, "implicit": implicit.__version__, "catboost": catboost.__version__},
    "events": len(events),
    "train": len(train_events),
    "validation": len(validation_events),
    "test": len(test_events),
    "users_in_train": train_matrix.shape[0],
    "items_in_train": train_matrix.shape[1],
})


## 1. Контракт событий и временное разбиение

Одна строка — факт лайка: пользователь `uid`, объект `item_id`, время `timestamp`, признак органического взаимодействия `is_organic`. Мы прогнозируем будущие лайки по прошлым. Поэтому граница времени общая для всех пользователей: модель не должна видеть события, которые в продукте ещё не произошли.

Проверьте границы, пересечения событий и доступность пользователей. Для основной оценки оставьте пользователей, у которых есть хотя бы пять лайков в train и хотя бы один лайк в validation и test. Пользователей без истории не удаляйте из отчёта: для них нужен fallback по популярности.

In [ ]:
def build_evaluation_users(train, validation, test):
    # TODO: верните DataFrame columns=[uid, train_events, validation_events, test_events, segment].
    # segment: cold (нет train), short (5–19 train), established (20+ train).
    raise NotImplementedError


evaluation_users = build_evaluation_users(train_events, validation_events, test_events)
eligible_users = evaluation_users.query("train_events >= 5 and validation_events >= 1 and test_events >= 1").uid.tolist()
split_audit = {
    "train_max": int(train_events.timestamp.max()),
    "validation_min": int(validation_events.timestamp.min()),
    "validation_max": int(validation_events.timestamp.max()),
    "test_min": int(test_events.timestamp.min()),
    "eligible_users": len(eligible_users),
}
split_audit


## 2. Популярность

Это не заглушка, а обязательная контрольная модель. Посчитайте число уникальных пользователей для каждого объекта только по train, отсортируйте объекты детерминированно и для каждого пользователя верните первые 20 ещё не просмотренных объектов. Измерьте время подготовки и среднее время ответа на пользователя.

In [ ]:
def fit_popularity(history: pd.DataFrame) -> pd.DataFrame:
    # TODO: верните columns=[item_id, score, rank]; score — число уникальных пользователей.
    raise NotImplementedError


def recommend_popularity(popularity: pd.DataFrame, history: pd.DataFrame, user_ids, k=20):
    # TODO: верните dict[uid, list[item_id]], исключив объекты из history этого пользователя.
    raise NotImplementedError


popularity_started = time.perf_counter()
popularity_model = fit_popularity(train_events)
popularity_fit_seconds = time.perf_counter() - popularity_started
popularity_recommendations = recommend_popularity(popularity_model, train_events, eligible_users, k=20)


## 3. Item-to-item по сходству взаимодействий

Представьте объект бинарным вектором пользователей. Близкими считаются объекты, которые лайкали похожие люди. Соберите `CosineRecommender(K=100)` из `implicit.nearest_neighbours`, обучите его на той же матрице и получите top-20. Объясните, зачем ограничивать число соседей и что произойдёт с редким объектом.

In [ ]:
from implicit.nearest_neighbours import CosineRecommender


def fit_item_to_item(user_item_matrix: sparse.csr_matrix):
    # TODO: обучите CosineRecommender(K=100) и верните модель вместе со временем fit.
    raise NotImplementedError


def recommend_implicit_model(model, user_item_matrix, user_ids, k=20):
    # TODO: вызовите model.recommend для известных пользователей, filter_already_liked_items=True.
    # Преобразуйте внутренние индексы объектов обратно в исходные item_id.
    raise NotImplementedError


item_to_item_model, item_to_item_fit_seconds = fit_item_to_item(train_matrix)
item_to_item_recommendations = recommend_implicit_model(
    item_to_item_model, train_matrix, eligible_users, k=20
)


## 4. Implicit ALS

ALS раскладывает разреженную матрицу взаимодействий на векторы пользователей и объектов. Здесь отсутствие лайка не является доказанным дизлайком: это неизвестное взаимодействие. Обучите `AlternatingLeastSquares` с `factors=32`, `regularization=0.05`, `iterations=10`, `random_state=42` и CPU. Не подбирайте параметры по test.

In [ ]:
from implicit.als import AlternatingLeastSquares


def fit_als(user_item_matrix: sparse.csr_matrix):
    # TODO: обучите закреплённую ALS и верните модель вместе со временем fit.
    raise NotImplementedError


als_model, als_fit_seconds = fit_als(train_matrix)
als_recommendations = recommend_implicit_model(als_model, train_matrix, eligible_users, k=20)


## 5. Общая оценка и стоимость

Для каждого пользователя релевантны его лайки из test. Считайте Recall@20 и NDCG@20 сначала по пользователям, затем усредняйте. Покрытие — доля уникальных рекомендованных объектов от каталога train. Измерьте среднее время формирования ответа. Все три модели должны оцениваться на одном наборе пользователей.

In [ ]:
def ranking_metrics(recommendations, targets: pd.DataFrame, catalog_size: int, k=20):
    # TODO: верните общий dict и per_user DataFrame.
    # Общий dict содержит recall_at_20, ndcg_at_20, catalog_coverage и evaluated_users.
    raise NotImplementedError


def timed_recommend(callable_):
    # TODO: верните результат и среднее время на пользователя в миллисекундах.
    raise NotImplementedError


model_runs = {
    "popularity": (popularity_recommendations, popularity_fit_seconds),
    "item_to_item": (item_to_item_recommendations, item_to_item_fit_seconds),
    "als": (als_recommendations, als_fit_seconds),
}
comparison_rows = []
per_user_reports = {}
for model_name, (recommendations, fit_seconds) in model_runs.items():
    # TODO: посчитайте метрики и добавьте одну строку comparison_rows.
    pass

comparison = pd.DataFrame(comparison_rows)
comparison


## 6. Новые пользователи, редкие объекты и решение

Покажите минимум два среза: пользователи с 5–19 событиями в train и цели, относящиеся к нижним 50% объектов по популярности train. Для пользователя без истории примените popularity fallback и явно укажите, что персонализации в нём нет.

Выберите первый production-бейзлайн. Назовите выигрыш, проигрыш, вычислительную цену и условие пересмотра решения. Если ALS выигрывает общий Recall, но хуже на короткой истории или заметно дороже, это должно быть видно в выводе.

In [ ]:
def build_slice_report(per_user_reports, train, test, evaluation_users):
    # TODO: верните таблицу model, slice, users, recall_at_20, ndcg_at_20.
    # Обязательные slice: all, short_history, rare_target.
    raise NotImplementedError


slice_report = build_slice_report(per_user_reports, train_events, test_events, evaluation_users)
cold_user_example = int(evaluation_users.sort_values(["train_events", "uid"]).iloc[0].uid)
cold_user_recommendations = recommend_popularity(popularity_model, train_events, [cold_user_example], k=20)

ENGINEERING_DECISION = {
    "ship_first": None,  # TODO: popularity | item_to_item | als
    "quality_evidence": None,
    "slice_risk": None,
    "compute_cost": None,
    "fallback": "popularity",
    "revisit_when": None,
}


In [ ]:
# Self-check: проверяет полноту работы, но не выбирает модель за вас.
assert split_audit["train_max"] < split_audit["validation_min"]
assert split_audit["validation_max"] < split_audit["test_min"]
assert len(eligible_users) >= 1_000
assert set(comparison.model) == {"popularity", "item_to_item", "als"}
assert comparison.recall_at_20.between(0, 1).all()
assert comparison.ndcg_at_20.between(0, 1).all()
assert comparison.catalog_coverage.between(0, 1).all()
assert (comparison.fit_seconds >= 0).all() and (comparison.mean_inference_ms >= 0).all()
for uid in eligible_users:
    seen = set(train_events.loc[train_events.uid == uid, "item_id"])
    for recommendations, _ in model_runs.values():
        assert not (seen & set(recommendations[uid][:20]))
assert {"all", "short_history", "rare_target"} <= set(slice_report.slice)
assert all(ENGINEERING_DECISION[key] is not None for key in [
    "ship_first", "quality_evidence", "slice_risk", "compute_cost", "revisit_when"
])
print(comparison.sort_values("ndcg_at_20", ascending=False))
display(slice_report)


## Что сдать

1. Заполненный notebook без удаления assertions.
2. Таблицу сравнения и таблицу срезов.
3. Короткое решение о первом выпуске: модель, fallback, стоимость и риск.
4. Пояснение, почему случайный split и вывод «ALS лучше» были бы недостаточны.

Не публикуйте выполненный notebook в репозитории ML Mentor: сохраните личную копию в Google Drive или выгрузите её локально.